# The Agent Loop — who decides when to stop

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emergtechinc/incident-triage-agent/blob/main/notebooks/01_the_agent_loop.ipynb)

**No API key needed. Nothing to install. Runs in about a minute.**

Everyone draws an agent as a cycle — perceive, reason, act, repeat. That picture leaves out the
only thing that matters: **who runs the cycle.**

By the end of this notebook you will have built the loop yourself, watched it investigate a real
incident, and watched a bound stop it.

## 0. Setup

We clone the project repo and use the code in it — including the parts your teammates have already
merged.

There is no API key in this notebook and no call to Anthropic. Instead we use a **scripted stand-in**
for the client (`src/triage/fake.py`) that replays responses we choose. That is not a shortcut: it is
how you test any system whose dependency is non-deterministic.

In [ ]:
!git clone -q https://github.com/emergtechinc/incident-triage-agent.git
%cd incident-triage-agent
import sys; sys.path.insert(0, "src")

from triage import fixtures, pricing
from triage.fake import FakeClient, says, wants_tool, wants_tools, truncated, refused

print("cloned and imported — no key, no network")
print("rates last verified:", pricing.RATES_TAKEN_ON)

## 1. The incident

Synthetic, and deliberately **contradictory**. Read it, then read the tool data underneath it.

In [ ]:
print(fixtures.INCIDENTS["moderate"], "\n")

print("checkout-api last deployed :", fixtures.DEPLOYS["checkout-api"][0]["minutes_ago"], "minutes ago")
print("payments-svc last deployed :", fixtures.DEPLOYS["payments-svc"][0]["minutes_ago"], "minutes ago")
print("checkout-api depends on    :", fixtures.DEPENDENCIES["checkout-api"])

**Notice the shape of the problem.** The service reporting trouble has not deployed in three days.
Something it *depends on* shipped 55 minutes ago.

**No single tool call finds that.** You have to ask about the service, notice its deploys are stale,
ask what it depends on, and then ask about *that*. Three questions, each one chosen because of the
last answer.

That is the only honest reason an agent needs a loop.

## 2. The model answers once and stops

Before writing a loop, watch what happens without one. We give the model the incident **and** the
tools, and call it exactly once.

In [ ]:
client = FakeClient([
    wants_tool("get_service_metrics", {"service": "checkout-api"}),
])

response = client.messages.create(
    model="claude-sonnet-5", max_tokens=1000,
    system=fixtures.SYSTEM_PROMPT,
    messages=[{"role": "user", "content": fixtures.INCIDENTS["moderate"]}],
    tools=[],
)

print("stop_reason:", response.stop_reason)
for block in response.content:
    if block.type == "tool_use":
        print("it wants  :", block.name, block.input)
        print("block id  :", block.id, " <- remember this")

print("\nIt asked for a tool and stopped. Nobody ran it.")
print("The investigation is frozen here until OUR code continues it.")

## 3. `stop_reason` is the whole mechanism

Four values. Each one means something different, and **three of them are not "keep going."**

In [ ]:
scenarios = {
    "finished normally":     says("The connection pool is exhausted."),
    "wants a tool":          wants_tool("get_dependencies", {"service": "checkout-api"}),
    "ran out of room":       truncated("The likely cause is that the connection"),
    "declined":              refused(),
}

for label, scripted in scenarios.items():
    c = FakeClient([scripted])
    r = c.messages.create(model="claude-sonnet-5", max_tokens=100,
                          system="s", messages=[{"role": "user", "content": "x"}])
    text = "".join(b.text for b in r.content if b.type == "text")
    print(f"  {label:<20} stop_reason = {r.stop_reason:<12} {('…' + text[-34:]) if text else ''}")

**Look at the third one.** It stopped mid-sentence because it ran out of room — and the text it
produced reads exactly like an answer.

This is the bug that ships everywhere:

```python
while response.stop_reason == "tool_use":   #  DON'T
```

That treats every other value as success. A half-sentence gets returned to a user as finished, and
**nothing errors and nothing logs.**

## 4. Build the loop

Now the whole thing. Read the comments — every numbered step is a decision, not plumbing.

> `results.py` here is **Gomathi's merged code** (ticket #4), and the cost arithmetic is
> **Naveen's** (ticket #1). You are running your own team's work.

In [ ]:
# Probe by CALLING it -- the import succeeds either way; only the call raises.
from triage import results as _r

try:
    _r.tool_result("probe", "x")
    tool_result, results_message = _r.tool_result, _r.results_message
    print("using the merged results.py — ticket #4, Gomathi's code")
except NotImplementedError:
    def tool_result(tool_use_id, content, *, is_error=False):
        b = {"type": "tool_result", "tool_use_id": tool_use_id, "content": content}
        if is_error: b["is_error"] = True
        return b
    def results_message(results):
        return {"role": "user", "content": results}
    print("ticket #4 not merged yet — using a local copy for this demo")

In [ ]:
import json


def run_tool(name, args):
    """Stands in for ticket #2. Returns JSON, and never raises."""
    service = args.get("service", "")
    if name == "get_service_metrics":
        return json.dumps(fixtures.METRICS.get(service, {"error": "unknown service"}))
    if name == "get_recent_deploys":
        return json.dumps(fixtures.DEPLOYS.get(service, {"error": "unknown service"}))
    if name == "get_dependencies":
        return json.dumps({"service": service,
                           "shares_infra_with": fixtures.DEPENDENCIES.get(service, [])})
    return json.dumps({"error": f"unknown tool {name}"})


def run(client, incident, *, model="claude-sonnet-5", max_turns=8):
    messages = [{"role": "user", "content": incident}]
    total_cost = 0.0

    for turn in range(1, max_turns + 1):
        response = client.messages.create(
            model=model, max_tokens=1500,
            system=fixtures.SYSTEM_PROMPT, messages=messages, tools=[],
        )
        total_cost += pricing.cost_of_response(model, response.usage)   # Naveen's code

        # 1. The assistant's blocks go back VERBATIM -- tool_use blocks included.
        #    Drop them and the next message answers a question that is no longer there.
        messages.append({"role": "assistant", "content": response.content})

        print(f"  turn {turn}: stop_reason={response.stop_reason:<10} running ${total_cost:.6f}")

        # 2. EVERY terminal value handled -- not just tool_use.
        if response.stop_reason != "tool_use":
            answer = "".join(b.text for b in response.content if b.type == "text")
            status = {"end_turn": "COMPLETED",
                      "max_tokens": "TRUNCATED -- not an answer",
                      "refusal": "DECLINED"}.get(response.stop_reason, "UNKNOWN")
            return status, answer.strip(), total_cost

        # 3. Run every tool it asked for.
        blocks = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            out = run_tool(block.name, block.input)
            print(f"        -> {block.name}({block.input}) => {out[:60]}")
            blocks.append(tool_result(block.id, out))

        # 4. ALL results, ONE message. Split them and the model stops asking in parallel.
        messages.append(results_message(blocks))

    return "BOUNDED -- hit max_turns", "", total_cost


print("loop defined")

## 5. Watch it investigate

The script below is what a model plausibly does with this incident: check the service, find its
deploys are stale, ask what it depends on, then check that.

In [ ]:
client = FakeClient([
    wants_tool("get_service_metrics",  {"service": "checkout-api"},  tool_id="t1"),
    wants_tool("get_recent_deploys",   {"service": "checkout-api"},  tool_id="t2"),
    wants_tool("get_dependencies",     {"service": "checkout-api"},  tool_id="t3"),
    wants_tool("get_recent_deploys",   {"service": "payments-svc"},  tool_id="t4"),
    says("checkout-api degraded ~45 minutes ago but has not deployed in three days. "
         "It shares a connection pool with payments-svc, which shipped a pool-sizing "
         "change 55 minutes ago. Start there."),
])

status, answer, cost = run(client, fixtures.INCIDENTS["moderate"])

print(f"\n  {status}\n")
print(f"  {answer}\n")
print(f"  total: ${cost:.6f}")

**Four turns to reach a conclusion no single call could.** And notice the input tokens climbing each
turn — the API remembers nothing, so you resend the whole conversation every time. Turn four pays
for turns one through three again.

That is why cost is the bound that matters, and why an iteration cap alone is not enough.

## 6. Watch a bound stop it

Same loop, `max_turns=2`. The agent had not finished — and the output says so rather than dressing a
partial result up as an answer.

In [ ]:
client = FakeClient([wants_tool("get_dependencies", {"service": "checkout-api"})] * 10)

status, answer, cost = run(client, fixtures.INCIDENTS["moderate"], max_turns=2)
print(f"\n  {status}")
print(f"  spent ${cost:.6f} and produced no answer.")
print("\n  In production this is where a human gets paged -- not where we guess.")

## What you just built, and what is left

The loop above is about twenty lines. **Everything hard is around it:**

| | Ticket |
|---|---|
| Three limits — turns, dollars, seconds | **#3** — Lakshmi |
| Tool schemas and descriptions that stop misrouting | **#2** — Sowmya |
| The loop itself, properly, with all four terminal values | **#5** — open |
| Making it a program you can run | **#6** — open |
| Letting the agent say *"I don't know, and here's what would tell me"* | **#7** — open |

The two that are already merged — pricing and results — are the code you just ran.

### The question worth taking away

Not *"does it work."*

> **What stops it, and what does the worst case cost?**

If those two have no answer, you do not have an agent. You have a loop.